# STEP 8 — Mask-Based Anomaly Segmentation Baselines (EoMT)

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

TOKEN = userdata.get('GITHUB_TOKEN')
REPO  = "MaskArchitectureAnomaly_CourseProject"

!git clone --branch debugging_step_4 https://{TOKEN}@github.com/filoppos/MaskArchitectureAnomaly_CourseProject.git /content/project

In [ ]:
!git config --global user.email "fil.manca@stud.uniroma3.it"
!git config --global user.name "filoppos"

In [ ]:
%cd /content/project/eomt

In [ ]:
!pip install -q -r requirements.txt

### A) Setup — import, path, transform, metriche (riusati dallo Step 7)

In [ ]:
import os, sys, glob, random, time
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize
from sklearn.metrics import average_precision_score, roc_curve

def fpr_at_95_tpr(scores, labels):
    fpr, tpr, _ = roc_curve(labels, scores)
    idx = np.searchsorted(tpr, 0.95)
    return float(fpr[min(idx, len(fpr) - 1)])

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

DRIVE_BASE   = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project"
DATASETS_DIR = os.path.join(DRIVE_BASE, "Validation_Dataset")

DEVICE = 0

def pil_to_eomt_tensor(pil_rgb):
    '''PIL RGB -> tensore uint8 [C,H,W] 0-255 a risoluzione nativa.'''
    arr = np.array(pil_rgb)
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

target_transform = Compose([Resize((1024, 2048), Image.NEAREST)])

print("✓ Setup A completato")
print(f"  DATASETS : {DATASETS_DIR}")
print("  Input EoMT: uint8 0-255 a risoluzione nativa; resize a model.img_size nel forward")

In [ ]:
DATASET_CONFIGS = {
    "SMIYC RA-21": {"folder": "RoadAnomaly21",     "pattern": "images/*.png"},
    "SMIYC RO-21": {"folder": "RoadObsticle21",    "pattern": "images/*.webp"},
    "FS L&F":      {"folder": "FS_LostFound_full",  "pattern": "images/*.png"},
    "FS Static":   {"folder": "fs_static",          "pattern": "images/*.jpg"},
    "Road Anomaly":{"folder": "RoadAnomaly",        "pattern": "images/*.jpg"},
}

def get_gt_path(img_path):
    gt = img_path.replace("images", "labels_masks")
    base, ext = os.path.splitext(gt)
    if ext.lower() in (".webp", ".jpg", ".jpeg"):
        gt = base + ".png"
    return gt

def fix_gt(ood_gts, dataset_name):
    folder = DATASET_CONFIGS[dataset_name]["folder"]
    if "RoadAnomaly" in folder:
        ood_gts = np.where(ood_gts == 2, 1, ood_gts)
    if "Streethazard" in folder:
        ood_gts = np.where(ood_gts == 14, 255, ood_gts)
        ood_gts = np.where(ood_gts  < 20,   0, ood_gts)
        ood_gts = np.where(ood_gts == 255,  1, ood_gts)
    return ood_gts

print("=== Verifica dataset ===")
for ds_name, cfg in DATASET_CONFIGS.items():
    folder_path = os.path.join(DATASETS_DIR, cfg["folder"])
    found = []
    for pat in [cfg["pattern"], "images/*.png", "images/*.jpg", "images/*.webp"]:
        found = glob.glob(os.path.join(folder_path, pat))
        if found:
            cfg["pattern"] = pat
            break
    status = f"{len(found):4d} immagini" if found else "⚠ NESSUNA IMMAGINE"
    print(f"  {'✓' if found else '✗'}  {ds_name:15s}  {status:20s}  [{cfg['pattern']}]")

### B) Costruzione del modello EoMT (parametrica per task)

In [ ]:
import yaml, importlib, warnings
from lightning import seed_everything
from torch.amp.autocast_mode import autocast

seed_everything(0, verbose=False)
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module`.*",
)

DATA_PATH = DRIVE_BASE

CONFIG_PATHS = {
    "cityscapes": "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml",
    "coco":       "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml",
}

def _load_cfg(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

def build_eomt(task):
    '''Crea un modello EoMT (in eval) per il task indicato. Ritorna (model, num_classes).'''
    assert task in CONFIG_PATHS, f"task sconosciuto: {task}"
    active_config = _load_cfg(CONFIG_PATHS[task])

    dm_name, dm_cls_name = active_config["data"]["class_path"].rsplit(".", 1)
    dm_cls = getattr(importlib.import_module(dm_name), dm_cls_name)
    dm_kwargs = active_config["data"].get("init_args", {})
    model_data = dm_cls(path=DATA_PATH, batch_size=1, num_workers=0,
                        check_empty_targets=False, **dm_kwargs)

    enc_cfg = active_config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_cls_name = enc_cfg["class_path"].rsplit(".", 1)
    enc_cls = getattr(importlib.import_module(enc_mod), enc_cls_name)
    encoder = enc_cls(img_size=model_data.img_size, **enc_cfg.get("init_args", {}))

    net_cfg = active_config["model"]["init_args"]["network"]
    net_mod, net_cls_name = net_cfg["class_path"].rsplit(".", 1)
    net_cls = getattr(importlib.import_module(net_mod), net_cls_name)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = net_cls(masked_attn_enabled=False,
                      num_classes=model_data.num_classes,
                      encoder=encoder, **net_kwargs)

    lit_mod, lit_cls_name = active_config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_mod), lit_cls_name)
    model_kwargs = {k: v for k, v in active_config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in active_config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = active_config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(img_size=model_data.img_size,
                    num_classes=model_data.num_classes,
                    network=network, **model_kwargs).eval().to(DEVICE)
    print(f"  ↳ build task={task}: img_size={model_data.img_size}, num_classes={model_data.num_classes}")
    return model, model_data.num_classes

def load_eomt_weights(model, weights_path):
    '''Carica i pesi nel modello. Gestisce sia state_dict puri sia checkpoint Lightning.'''
    sd = torch.load(weights_path, map_location=f"cuda:{DEVICE}", weights_only=False)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    info = model.load_state_dict(sd, strict=False)
    print(f"  ↳ pesi caricati da {os.path.basename(weights_path)} "
          f"(missing={len(info.missing_keys)}, unexpected={len(info.unexpected_keys)})")
    return model

### C) Da EoMT alla mappa per-pixel `[C,H,W]`

In [ ]:
def eomt_per_pixel_logits(model, img_tensor):
    short = min(model.img_size)
    _, H, W = img_tensor.shape
    scale = short / min(H, W)
    new_h, new_w = round(H * scale), round(W * scale)
    img_r = F.interpolate(img_tensor[None].float(), size=(new_h, new_w),
                          mode="bilinear", align_corners=False)[0]
    img_r = img_r.round().clamp(0, 255).to(torch.uint8)
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img_r.to(DEVICE)]
        img_sizes = [im.shape[-2:] for im in imgs]
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        crop_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
    return logits[0].float().cpu().numpy()

### D) Scoring functions: MSP · MaxLogit · MaxEntropy · RbA

In [ ]:
def anomaly_score_msp(pixel_logits):
    shifted = pixel_logits - pixel_logits.max(axis=0, keepdims=True)
    exp = np.exp(shifted); probs = exp / exp.sum(axis=0, keepdims=True)
    return 1.0 - probs.max(axis=0)

def anomaly_score_maxlogit(pixel_logits):
    return 1.0 - pixel_logits.max(axis=0)

def anomaly_score_maxentropy(pixel_logits):
    shifted = pixel_logits - pixel_logits.max(axis=0, keepdims=True)
    exp = np.exp(shifted); probs = exp / exp.sum(axis=0, keepdims=True)
    entropy = -(probs * np.log(probs + 1e-8)).sum(axis=0)
    return entropy / np.log(pixel_logits.shape[0])

def anomaly_score_rba(pixel_logits):
    """RbA: somma SULLE CLASSI dei punteggi per-pixel, negata."""
    return -pixel_logits.sum(axis=0)

SCORING_METHODS = {
    "MSP":        anomaly_score_msp,
    "MaxLogit":   anomaly_score_maxlogit,
    "MaxEntropy": anomaly_score_maxentropy,
    "RbA":        anomaly_score_rba,
}

_d = np.random.randn(19, 64, 128).astype(np.float32)
for n, fn in SCORING_METHODS.items():
    s = fn(_d); print(f"  {n:11s} shape={s.shape} min={s.min():.3f} max={s.max():.3f}")
print("\n✓ Scoring functions pronte")

In [ ]:
COCO_TO_CITYSCAPES = {
    0:11, 1:18, 2:13, 3:17, 5:15, 6:16, 7:14, 9:6, 11:7,
    100:0, 123:1, 129:2, 91:2, 82:2, 101:2, 107:2, 86:2, 118:2,
    115:2, 114:2, 109:3, 110:3, 111:3, 112:3, 131:3, 117:4, 94:4,
    116:8, 88:8, 90:9, 96:9, 97:9, 98:9, 102:9, 126:9, 125:9,
    130:9, 105:9, 119:10, 92:5, 10:5,
}

def project_coco_to_cityscapes(pixel_logits, num_cs=19):
    """[133,H,W] -> [19,H,W]: somma gli pseudo-score COCO nelle 19 classi Cityscapes."""
    out = np.zeros((num_cs,) + pixel_logits.shape[1:], dtype=pixel_logits.dtype)
    for coco_idx, cs_idx in COCO_TO_CITYSCAPES.items():
        out[cs_idx] += pixel_logits[coco_idx]
    return out

print("✓ Proiezione COCO→19 pronta (", len(COCO_TO_CITYSCAPES), "classi mappate )")

### E) I tre checkpoint EoMT + i loro mIoU

In [ ]:
WEIGHTS_DIR = os.path.join(DRIVE_BASE, "weights")

CHECKPOINTS = {
    "EoMT (fine-tuned)": ("cityscapes", os.path.join(WEIGHTS_DIR, "eomt_finetuned.pth")),
    "EoMT (COCO)":       ("coco",       os.path.join(WEIGHTS_DIR, "eomt_coco.bin")),
    "EoMT (Cityscapes)": ("cityscapes", os.path.join(WEIGHTS_DIR, "eomt_cityscapes.bin")),
}

CHECKPOINT_MIOU = {
    "EoMT (COCO)":0.5452       ,
    "EoMT (Cityscapes)": 0.814,
    "EoMT (fine-tuned)": 0.75159,
}

### F) Loop di valutazione — forward pass unico per baseline + temperature scaling

In [ ]:
EVAL_HW          = (512, 1024)
LOGITS_CACHE_DIR = os.path.join(DRIVE_BASE, "step8_logits_cache")
os.makedirs(LOGITS_CACHE_DIR, exist_ok=True)

results      = {}
total_start  = time.time()

for ckpt_label, (task, wpath) in CHECKPOINTS.items():
    print("\n" + "#"*70)
    print(f"# CHECKPOINT: {ckpt_label}   (task={task})")
    print("#"*70)

    if not os.path.exists(wpath):
        print(f"⚠ Pesi non trovati: {wpath} → skip"); continue

    ckpt_key = ckpt_label.replace(" ","_").replace("(","").replace(")","")
    model_loaded = False
    model        = None

    results[ckpt_label] = {}

    for ds_name, cfg in DATASET_CONFIGS.items():
        ds_key    = ds_name.replace(" ", "_")
        cache_dir = os.path.join(LOGITS_CACHE_DIR, ckpt_key, ds_key)
        os.makedirs(cache_dir, exist_ok=True)

        folder_path = os.path.join(DATASETS_DIR, cfg["folder"])
        img_paths   = sorted(glob.glob(os.path.join(folder_path, cfg["pattern"])))
        if not img_paths:
            print(f"  ⚠ {ds_name}: nessuna immagine, skip"); continue

        print(f"\n  === {ds_name} ({len(img_paths)} img) ===")
        t0 = time.time()

        scores_flat = {m: [] for m in SCORING_METHODS}
        labels_flat = []
        skipped     = 0

        for path in img_paths:
            pathGT = get_gt_path(path)
            if not os.path.exists(pathGT):
                skipped += 1; continue

            ood_gts = fix_gt(np.array(Image.open(pathGT)), ds_name)
            if ood_gts.shape != EVAL_HW:
                ood_gts = np.array(Image.fromarray(ood_gts).resize(
                    (EVAL_HW[1], EVAL_HW[0]), Image.NEAREST))
            valid = ood_gts != 255
            if valid.sum() == 0 or 1 not in np.unique(ood_gts[valid]):
                skipped += 1; continue

            fname    = os.path.splitext(os.path.basename(path))[0] + ".npy"
            npy_path = os.path.join(cache_dir, fname)

            if os.path.exists(npy_path):
                pixel_logits = np.load(npy_path).astype(np.float32)
            else:
                if not model_loaded:
                    model, _ = build_eomt(task)
                    model     = load_eomt_weights(model, wpath)
                    model.eval()
                    model_loaded = True

                img          = pil_to_eomt_tensor(Image.open(path).convert("RGB"))
                pixel_logits = eomt_per_pixel_logits(model, img)

                if task == "coco":
                    pixel_logits = project_coco_to_cityscapes(pixel_logits)

                pl_t = torch.from_numpy(np.ascontiguousarray(pixel_logits))[None].float()
                pixel_logits = F.interpolate(pl_t, size=EVAL_HW,
                                             mode="bilinear", align_corners=False)[0].numpy()
                np.save(npy_path, pixel_logits.astype(np.float16))

            labels_flat.append(ood_gts[valid].astype(np.int32))

            for m_name, fn in SCORING_METHODS.items():
                sc = fn(pixel_logits)
                scores_flat[m_name].append(sc[valid].astype(np.float32))

            del pixel_logits

        print(f"  → {time.time()-t0:.0f}s  ({len(labels_flat)} usate, {skipped} skip)")

        if not labels_flat:
            print("  ⚠ nessuna immagine valida, skip"); continue

        labels_cat = np.concatenate(labels_flat)
        results[ckpt_label][ds_name] = {}

        for m_name, score_lists in scores_flat.items():
            if not score_lists: continue
            scores_cat = np.concatenate(score_lists)
            auprc = average_precision_score(labels_cat, scores_cat)
            fpr   = fpr_at_95_tpr(scores_cat, labels_cat)
            results[ckpt_label][ds_name][m_name] = {"auprc": auprc, "fpr95": fpr}
            print(f"    {m_name:11s} AUPRC={auprc*100:6.2f}%   FPR95={fpr*100:6.2f}%")

    if model is not None:
        del model; torch.cuda.empty_cache()

print(f"\n✓ Completato in {(time.time()-total_start)/60:.1f} min")
print(f"  Logit salvati in: {LOGITS_CACHE_DIR}")

### G) Tabella combinata (formato progetto) + salvataggio CSV

In [ ]:
import pandas as pd

DATASETS_ORDER = ["SMIYC RA-21", "SMIYC RO-21", "FS L&F", "FS Static", "Road Anomaly"]

def fmt(v):
    return "—" if (v is None or v != v) else f"{v*100:.2f}"

rows = []

erfnet_csv = os.path.join(DRIVE_BASE, "step7_erfnet_results.csv")
ERFNET_MIOU = 69.7
if os.path.exists(erfnet_csv):
    df7 = pd.read_csv(erfnet_csv)
    for i, (_, r) in enumerate(df7.iterrows()):
        row = {"Model": "ERFNet" if i == 0 else "",
               "mIoU": ERFNET_MIOU if i == 0 else "",
               "Method": r["Method"]}
        for ds in DATASETS_ORDER:
            row[f"{ds} AuPRC"] = r.get(f"{ds}_AuPRC", "—")
            row[f"{ds} FPR95"] = r.get(f"{ds}_FPR95", "—")
        rows.append(row)
else:
    print("ℹ CSV Step 7 non trovato — blocco ERFNet lasciato vuoto (# TODO).")

EOMT_METHODS = ["MSP", "MaxLogit", "MaxEntropy", "RbA"]
for ckpt_label in CHECKPOINTS:
    if ckpt_label not in results:
        continue
    for i, method in enumerate(EOMT_METHODS):
        row = {"Model": ckpt_label if i == 0 else "",
               "mIoU":  (CHECKPOINT_MIOU.get(ckpt_label) if i == 0 else ""),
               "Method": method}
        for ds in DATASETS_ORDER:
            cell = results[ckpt_label].get(ds, {}).get(method)
            row[f"{ds} AuPRC"] = fmt(cell["auprc"]) if cell else "—"
            row[f"{ds} FPR95"] = fmt(cell["fpr95"]) if cell else "—"
        rows.append(row)

df = pd.DataFrame(rows)
pd.set_option("display.max_columns", None, "display.width", 200)
print(df.to_string(index=False))
print("\nValori AuPRC/FPR95 in %.  AuPRC↑ meglio · FPR95↓ meglio.")

out_csv = os.path.join(DRIVE_BASE, "step8_eomt_results.csv")
df.to_csv(out_csv, index=False)
print(f"\n✓ Salvato: {out_csv}")

### H) Visualizzazione qualitativa delle anomaly map

In [ ]:
import matplotlib.pyplot as plt

VIZ_CKPT = "EoMT (Cityscapes)"
VIZ_SAVE = os.path.join(DRIVE_BASE, "step8_viz")
os.makedirs(VIZ_SAVE, exist_ok=True)

task, wpath = CHECKPOINTS[VIZ_CKPT]
viz_model, _ = build_eomt(task)
viz_model = load_eomt_weights(viz_model, wpath); viz_model.eval()

def visualize_one(ds_name, cfg, n=2):
    folder = os.path.join(DATASETS_DIR, cfg["folder"])
    img_paths = sorted(glob.glob(os.path.join(folder, cfg["pattern"])))[:n]
    if not img_paths:
        return
    fig, axes = plt.subplots(n, 6, figsize=(26, 4*n))
    if n == 1:
        axes = axes[np.newaxis, :]
    for ax, t in zip(axes[0], ["Input", "GT", "MSP", "MaxLogit", "MaxEntropy", "RbA"]):
        ax.set_title(t, fontsize=11, fontweight="bold")
    for r, path in enumerate(img_paths):
        pil = Image.open(path).convert("RGB")
        img = pil_to_eomt_tensor(pil)
        result_np = eomt_per_pixel_logits(viz_model, img)

        axes[r, 0].imshow(pil.resize((1024, 512))); axes[r, 0].axis("off")
        pathGT = get_gt_path(path)
        if os.path.exists(pathGT):
            ood = fix_gt(np.array(target_transform(Image.open(pathGT))), ds_name)
            axes[r, 1].imshow(np.where(ood == 255, 0.5, ood.astype(float)),
                              cmap="RdYlGn_r", vmin=0, vmax=1)
        axes[r, 1].axis("off")
        for col, (name, fn) in enumerate(SCORING_METHODS.items(), start=2):
            s = fn(result_np); s = (s - s.min()) / (s.max() - s.min() + 1e-8)
            im = axes[r, col].imshow(s, cmap="inferno", vmin=0, vmax=1)
            axes[r, col].axis("off"); plt.colorbar(im, ax=axes[r, col], fraction=0.03)
    fig.suptitle(f"{VIZ_CKPT} — {ds_name}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    p = os.path.join(VIZ_SAVE, f"{VIZ_CKPT}_{ds_name}".replace(" ", "_") + ".png")
    plt.savefig(p, dpi=130, bbox_inches="tight"); print("  salvata:", p); plt.show()

for ds_name, cfg in DATASET_CONFIGS.items():
    print(f"\n--- {ds_name} ---")
    visualize_one(ds_name, cfg, n=2)

### J) Temperature Scaling — sweep della temperatura (no GPU)

In [ ]:
TEMPS = [0.1, 0.25, 0.5, 0.75, 1.0, 1.1, 1.5, 2.0, 3.0, 5.0, 8.0, 10.0]

def msp_score_T(pixel_logits, T):
    logits_T = pixel_logits / T
    shifted  = logits_T - logits_T.max(axis=0, keepdims=True)
    exp      = np.exp(shifted)
    probs    = exp / exp.sum(axis=0, keepdims=True)
    return (1.0 - probs.max(axis=0)).astype(np.float32)

def entropy_score_T(pixel_logits, T):
    logits_T = pixel_logits / T
    shifted  = logits_T - logits_T.max(axis=0, keepdims=True)
    exp      = np.exp(shifted)
    probs    = exp / exp.sum(axis=0, keepdims=True)
    entropy  = -(probs * np.log(probs + 1e-8)).sum(axis=0)
    return (entropy / np.log(pixel_logits.shape[0])).astype(np.float32)

temp_results = {}

for ckpt_label, (task, _) in CHECKPOINTS.items():
    ckpt_key = ckpt_label.replace(" ","_").replace("(","").replace(")","")
    temp_results[ckpt_label] = {}

    print(f"\n{'='*60}")
    print(f"  Sweep temperatura: {ckpt_label}")
    print(f"{'='*60}")

    for ds_name, cfg in DATASET_CONFIGS.items():
        ds_key    = ds_name.replace(" ", "_")
        cache_dir = os.path.join(LOGITS_CACHE_DIR, ckpt_key, ds_key)
        if not os.path.exists(cache_dir):
            continue

        folder_path = os.path.join(DATASETS_DIR, cfg["folder"])
        img_paths   = sorted(glob.glob(os.path.join(folder_path, cfg["pattern"])))

        all_logits, all_labels = [], []
        for path in img_paths:
            pathGT = get_gt_path(path)
            if not os.path.exists(pathGT): continue

            fname    = os.path.splitext(os.path.basename(path))[0] + ".npy"
            npy_path = os.path.join(cache_dir, fname)
            if not os.path.exists(npy_path): continue

            ood_gts = fix_gt(np.array(Image.open(pathGT)), ds_name)
            if ood_gts.shape != EVAL_HW:
                ood_gts = np.array(Image.fromarray(ood_gts).resize(
                    (EVAL_HW[1], EVAL_HW[0]), Image.NEAREST))
            valid = ood_gts != 255
            if valid.sum() == 0 or 1 not in np.unique(ood_gts[valid]): continue

            pl = np.load(npy_path).astype(np.float32)
            all_logits.append((pl, valid))
            all_labels.append(ood_gts[valid].astype(np.int32))

        if not all_labels: continue

        labels_cat = np.concatenate(all_labels)
        temp_results[ckpt_label][ds_name] = {}
        print(f"  {ds_name:15s}: {len(all_labels)} immagini, sweep {len(TEMPS)} temperature...")

        for T in TEMPS:
            msp_scores, ent_scores = [], []
            for pl, valid in all_logits:
                msp_scores.append(msp_score_T(pl, T)[valid])
                ent_scores.append(entropy_score_T(pl, T)[valid])

            msp_cat = np.concatenate(msp_scores)
            ent_cat = np.concatenate(ent_scores)

            temp_results[ckpt_label][ds_name][T] = {
                "MSP-T": {
                    "auprc": float(average_precision_score(labels_cat, msp_cat)),
                    "fpr95": float(fpr_at_95_tpr(msp_cat, labels_cat)),
                },
                "MaxEntropy-T": {
                    "auprc": float(average_precision_score(labels_cat, ent_cat)),
                    "fpr95": float(fpr_at_95_tpr(ent_cat, labels_cat)),
                },
            }

print("\n✓ Sweep completato")

### K) Temperature Scaling — temperatura ottimale e tabella risultati

In [ ]:
def best_temperature(temp_results, ckpt_label, method, metric="auprc"):
    sign    = 1 if metric == "auprc" else -1
    best_T, best_val = None, -np.inf
    all_T   = sorted({T for ds in temp_results[ckpt_label].values() for T in ds})
    for T in all_T:
        vals = [sign * temp_results[ckpt_label][ds][T][method][metric]
                for ds in temp_results[ckpt_label] if T in temp_results[ckpt_label][ds]
                and method in temp_results[ckpt_label][ds][T]]
        if vals and np.mean(vals) > best_val:
            best_val = np.mean(vals); best_T = T
    return best_T

print("=" * 60)
print("  TEMPERATURA OTTIMALE (massimizza AUPRC medio sui dataset)")
print("=" * 60)

best_T_table = {}
for ckpt_label in CHECKPOINTS:
    if ckpt_label not in temp_results or not temp_results[ckpt_label]: continue
    best_T_table[ckpt_label] = {}
    for method in ["MSP-T", "MaxEntropy-T"]:
        T_opt = best_temperature(temp_results, ckpt_label, method)
        best_T_table[ckpt_label][method] = T_opt
        print(f"  {ckpt_label:25s}  {method:14s} → T* = {T_opt}")

print("\n--- AUPRC per ogni T (media sui dataset) ---")
for ckpt_label in CHECKPOINTS:
    if ckpt_label not in temp_results or not temp_results[ckpt_label]: continue
    print(f"\n  {ckpt_label}")
    print(f"  {'T':>6}  {'MSP-T':>12}  {'Entropy-T':>12}")
    all_T = sorted({T for ds in temp_results[ckpt_label].values() for T in ds})
    for T in all_T:
        msp_v = [temp_results[ckpt_label][ds][T]["MSP-T"]["auprc"]
                 for ds in temp_results[ckpt_label] if T in temp_results[ckpt_label][ds]]
        ent_v = [temp_results[ckpt_label][ds][T]["MaxEntropy-T"]["auprc"]
                 for ds in temp_results[ckpt_label] if T in temp_results[ckpt_label][ds]]
        m = " ←" if T == best_T_table.get(ckpt_label,{}).get("MSP-T") else "  "
        e = " ←" if T == best_T_table.get(ckpt_label,{}).get("MaxEntropy-T") else "  "
        print(f"  {T:>6.2f}  {np.mean(msp_v)*100:>11.2f}%{m}  {np.mean(ent_v)*100:>11.2f}%{e}")

import pandas as pd

DATASETS_ORDER = ["SMIYC RA-21", "SMIYC RO-21", "FS L&F", "FS Static", "Road Anomaly"]
rows_ts = []

for ckpt_label in CHECKPOINTS:
    if ckpt_label not in best_T_table: continue
    for i, method in enumerate(["MSP-T", "MaxEntropy-T"]):
        T_opt = best_T_table[ckpt_label].get(method)
        row   = {"Model":  ckpt_label if i == 0 else "",
                 "mIoU":   CHECKPOINT_MIOU.get(ckpt_label, "—") if i == 0 else "",
                 "Method": f"{method} (T={T_opt})"}
        for ds in DATASETS_ORDER:
            cell = (temp_results[ckpt_label].get(ds, {})
                    .get(T_opt, {}).get(method, None))
            row[f"{ds} AuPRC"] = fmt(cell["auprc"]) if cell else "—"
            row[f"{ds} FPR95"] = fmt(cell["fpr95"]) if cell else "—"
        rows_ts.append(row)

df_ts = pd.DataFrame(rows_ts)
pd.set_option("display.max_columns", None, "display.width", 220)
print("\n" + "="*60)
print("  RISULTATI TEMPERATURE SCALING (T* per AUPRC)")
print("="*60)
print(df_ts.to_string(index=False))
print("\nAuPRC↑ meglio · FPR95↓ meglio")

out_csv = os.path.join(DRIVE_BASE, "step8_temperature_scaling_results.csv")
df_ts.to_csv(out_csv, index=False)
print(f"\n✓ Salvato: {out_csv}")

### L) Tabella stile paper — solo modello migliore (Cityscapes) + MSP a varie T

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

BEST_MODEL  = "EoMT (Cityscapes)"
METHOD      = "MSP-T"
TEMPS_TABLE = [0.5, 0.75, 1.0, 1.1, 5.0]
DATASETS_ORDER = ["SMIYC RA-21", "SMIYC RO-21", "FS L&F", "FS Static", "Road Anomaly"]

assert BEST_MODEL in temp_results and temp_results[BEST_MODEL], (
    f"'{BEST_MODEL}' non presente in temp_results: esegui prima il sweep (cella J).")

def t_label(T):
    "MSP (T = 0.5) / MSP (T = 1) ... interi senza decimali come nel paper."
    return f"MSP (T = {int(T) if float(T).is_integer() else T})"

def pct(v):
    return np.nan if (v is None or v != v) else round(v * 100, 2)

columns = pd.MultiIndex.from_tuples(
    [(ds, metric) for ds in DATASETS_ORDER for metric in ("AuPRC", "FPR95")])

data, index = [], []
for T in TEMPS_TABLE:
    index.append(t_label(T))
    row = []
    for ds in DATASETS_ORDER:
        cell = temp_results[BEST_MODEL].get(ds, {}).get(T, {}).get(METHOD)
        if cell is None:
            row += [np.nan, np.nan]
        else:
            row += [pct(cell["auprc"]), pct(cell["fpr95"])]
    data.append(row)

df_paper = pd.DataFrame(data, index=index, columns=columns)
df_paper.index.name = "Method"

pd.set_option("display.max_columns", None, "display.width", 220,
              "display.float_format", lambda x: f"{x:.2f}")
print("=" * 72)
print(f"  {BEST_MODEL} - MSP con temperature scaling  (AuPRC ^ - FPR95 v, in %)")
print("=" * 72)
display(df_paper)

out_csv = os.path.join(DRIVE_BASE, "step8_msp_temperature_table_bestmodel.csv")
df_paper.to_csv(out_csv)
print(f"\n* Salvato CSV  : {out_csv}")

def to_latex_paper(df):
    n = len(DATASETS_ORDER)
    L = [r"\begin{tabular}{l" + "cc" * n + "}", r"\toprule"]
    L.append(" & ".join([""] + [r"\multicolumn{2}{c}{%s}" % ds
                                 for ds in DATASETS_ORDER]) + r" \\")
    L.append(" & ".join([r"\textbf{Method}"] + ["AuPRC", "FPR95"] * n) + r" \\")
    L.append(r"\midrule")
    for idx, r in df.iterrows():
        cells = [idx] + [("%.2f" % v if v == v else "--") for v in r.values]
        L.append(" & ".join(cells) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(L)

latex_str = to_latex_paper(df_paper)
out_tex = os.path.join(DRIVE_BASE, "step8_msp_temperature_table_bestmodel.tex")
with open(out_tex, "w") as f:
    f.write(latex_str)
print(f"* Salvato LaTeX: {out_tex}\n")
print(latex_str)